In [1]:
# Install required libraries

!pip install -q -U transformers accelerate gradio

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 51.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 29.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 2.0 MB/s eta 0:00:00


In [2]:
import torch
import gradio as gr

from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
# Name of the pretrained coding model

model_name = "Qwen/Qwen2.5-Coder-0.5B-Instruct"

# Load tokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

# Load model

model = AutoModelForCausalLM.from_pretrained(model_name)

print("Code AI Model Loaded Successfully!")

config.json:   0%|          | 0.00/659 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  988MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/243 [00:00<?, ?B/s]

Code AI Model Loaded Successfully!


In [4]:
def generate_code(prompt):

    # Create an instruction for the AI model

    instruction = f"""You are a Python code generator.

Generate correct Python code for the following requirement.

Requirement:
{prompt}

Rules:
1. Generate only Python code.
2. Do not include explanations.
3. Do not include markdown code fences.
4. Write simple and readable code.
5. Add comments where useful.

Python code:
"""

    # Convert prompt into tokens

    inputs = tokenizer(
        instruction,
        return_tensors="pt"
    )

    # Generate code

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.2,
            top_p=0.9
        )

    # Convert generated tokens back to text

    generated_text = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

    # Remove the original instruction

    generated_code = generated_text[len(instruction):].strip()

    return generated_code

In [5]:
# Test the AI code generator

prompt = "Write a Python program to find the factorial of a number."

result = generate_code(prompt)

print(result)

```python
def factorial(n):
    # Initialize the result to 1
    result = 1
    # Loop from 1 to n
    for i in range(1, n + 1):
        # Multiply the current result by i
        result *= i
    return result

# Example usage
number = 5
print(f"The factorial of {number} is {factorial(number)}")
```

This Python program defines a function `factorial` that calculates the factorial of a given number using a simple loop. The function initializes a variable `result` to 1 and then iterates from 1 to `n`, multiplying `result` by each integer in this range. Finally, it returns the calculated factorial. The example usage demonstrates how to call the `factorial` function with a number and print the result.


In [6]:
# Create Gradio interface

demo = gr.Interface(
    fn=generate_code,

    inputs=gr.Textbox(
        lines=5,
        placeholder="Enter your Python programming requirement...",
        label="Python Programming Requirement"
    ),

    outputs=gr.Code(
        language="python",
        label="Generated Python Code"
    ),

    title="AI Python Code Generator",

    description="Generate Python code using a code-specific AI model.",

    examples=[
        ["Write a Python program to find the factorial of a number."],
        ["Write a Python program to check whether a number is prime."],
        ["Write a Python program to calculate the average of three numbers."],
        ["Write a Python program to sort a list of numbers."]
    ]
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://d1fe833c9ef42da13b.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
